# Otimização de rotas hospitalares

Este notebook reproduz o planejamento do cenário fictício. A análise começa por uma solução de referência, executa o algoritmo genético e examina viabilidade, ganho relativo e convergência.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from hospital_routes.io import load_problem
from hospital_routes.genetic import GAConfig, GeneticOptimizer
from hospital_routes.baselines import nearest_neighbor
from hospital_routes.reporting import GeminiReportGenerator, LocalReportGenerator, configured_gemini_key
from hospital_routes.visualization import save_route_map, save_convergence_plot
print(f'Kernel: {sys.executable}')
print('Gemini configurado:', bool(configured_gemini_key()))

## Cenário e configuração

A semente fixa torna a execução repetível. As penalidades de capacidade e autonomia são deliberadamente maiores que o custo de alguns quilômetros adicionais.

In [ ]:
problem = load_problem(ROOT / 'data' / 'deliveries.json')
config = GAConfig(population_size=120, generations=300, seed=42)
optimizer = GeneticOptimizer(problem, config)
baseline = nearest_neighbor(optimizer)
solution = optimizer.run()
{'entregas': len(problem.deliveries), 'veiculos': len(problem.vehicles), 'geracoes_executadas': len(optimizer.history)}

## Resultado comparativo

A comparação utiliza a mesma aptidão. Percentual positivo indica redução do custo composto em relação ao vizinho mais próximo.

In [ ]:
gain = 100 * (baseline.fitness - solution.fitness) / baseline.fitness
print(f'Fitness baseline: {baseline.fitness:.2f}')
print(f'Fitness GA: {solution.fitness:.2f}')
print(f'Redução: {gain:.2f}% | Distância: {solution.total_distance_km:.2f} km | Viável: {solution.feasible}')
[(problem.vehicles[i].id, [problem.deliveries[j].id for j in route]) for i, route in enumerate(solution.routes)]

## Visualizações

O gráfico permite verificar estabilização da população. O mapa HTML conserva a ordem das visitas e o retorno ao depósito.

In [ ]:
outputs = ROOT / 'outputs'
outputs.mkdir(exist_ok=True)
save_convergence_plot(optimizer.history, outputs / 'convergence.png')
save_route_map(problem, solution, outputs / 'routes_map.html')
from IPython.display import Image, display
display(Image(filename=outputs / 'convergence.png'))

## Instruções operacionais

A solução estruturada é enviada ao modelo Gemini configurado no projeto para produzir as instruções operacionais.

In [ ]:

report = GeminiReportGenerator().generate(problem, solution)
print('Relatório gerado pelo Gemini.')
print(report)